## **0. Settings**

In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))

## **1. Import Libraries**

In [2]:
import ast
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
from tqdm.notebook import tqdm
from sklearn.model_selection import GroupShuffleSplit
from src.training import dataset, data_loader, model

## **2. Get the Data**

In [3]:
it_ai_human_df = pd.read_csv('../data/preprocessing/it_ai_human_preprocessing_data.csv')
it_ai_human_df.head()

,cleaned_text,cleaned_original_text,tokens,input_ids,attention_mask,generated_type,label
0,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,NaN,"{'input_ids': tensor([[ 0, 2470, 621, ...,...","[0, 2470, 621, 447, 275, 4189, 1377, 72, 905, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",human,0
1,"Trong Web3, khái niệm về danh tính đang được x...",NaN,"{'input_ids': tensor([[ 0, 1222, 7572, ...,...","[0, 1222, 7572, 22, 15, 4401, 2036, 425, 1134,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",generated,1
2,**Hướng dẫn chi tiết cách kiểm tra phiên bản M...,Không chỉ riêng mình bạn mà nhiều người dùng k...,"{'input_ids': tensor([[ 0, 12322, 8355, ....","[0, 12322, 8355, 925, 907, 993, 677, 977, 818,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",rewrited,1
3,Một chiến dịch tấn công mạng tinh vi mới đang ...,NaN,"{'input_ids': tensor([[ 0, 1591, 823, ...,...","[0, 1591, 823, 797, 1354, 414, 1270, 1197, 851...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",human,0
4,Để có thể xem lại những bài viết đã đăng trên ...,Để xem lại tin Facebook đã đăng thì bạn cần ph...,"{'input_ids': tensor([[ 0, 2574, 293, ...,...","[0, 2574, 293, 404, 1183, 516, 417, 1115, 1410...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",rewrited,1


## **3. Create Group ID**

In [4]:
# Create group ID to avoid leakaging data
it_ai_human_df['group_key'] = np.where(
    it_ai_human_df['generated_type'] == 'rewrited',
    it_ai_human_df['cleaned_original_text'],
    it_ai_human_df['cleaned_text']
)

mask = (it_ai_human_df['generated_type'] == 'generated')
it_ai_human_df.loc[mask, 'group_key'] = 'generated_' + it_ai_human_df.index[mask].astype(str)

it_ai_human_df['group_id'] = pd.factorize(it_ai_human_df['group_key'])[0]

In [5]:
# Check first 5 rows
it_ai_human_df.sort_values('group_id')[['cleaned_text', 'cleaned_original_text', 'group_id', 'generated_type']][:5]

,cleaned_text,cleaned_original_text,group_id,generated_type
0,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,NaN,0,human
2464,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,Sự xuất hiện của iPhone 17e đánh dấu bước tiếp...,0,rewrited
1,"Trong Web3, khái niệm về danh tính đang được x...",NaN,1,generated
2,**Hướng dẫn chi tiết cách kiểm tra phiên bản M...,Không chỉ riêng mình bạn mà nhiều người dùng k...,2,rewrited
4007,Không chỉ riêng mình bạn mà nhiều người dùng k...,NaN,2,human


## **4. Split Groups**

In [6]:
# Prepare data
X = it_ai_human_df['tokens']
y = it_ai_human_df['label']
groups = it_ai_human_df['group_id']

In [7]:
SEED = 42

# Split 70 / 15 / 15
gss = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
train_idx, temp_idx = next(gss.split(X, y, groups))

In [8]:
X_temp = X.iloc[temp_idx]
y_temp = y.iloc[temp_idx]
groups_temp = groups.iloc[temp_idx]

In [9]:
gss = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED)
val_idx_local, test_idx_local = next(gss.split(X_temp, y_temp, groups_temp))

In [10]:
val_idx = temp_idx[val_idx_local]
test_idx = temp_idx[test_idx_local]

In [11]:
# Add new col split
it_ai_human_df['split'] = ''
it_ai_human_df.loc[train_idx, 'split'] = 'train'
it_ai_human_df.loc[val_idx, 'split'] = 'val'
it_ai_human_df.loc[test_idx, 'split'] = 'test'

In [12]:
# Check split value counts
it_ai_human_df['split'].value_counts()

split
train    3502
test      766
val       732
Name: count, dtype: int64

In [13]:
# Check whether data leakaged
it_ai_human_df.groupby('group_id')['split'].nunique().max()

np.int64(1)

## **5. Create Pytorch Dataset**

In [14]:
it_ai_human_df = it_ai_human_df.drop(columns='tokens')

In [15]:
# Convert input_ids and attention_mask to list
it_ai_human_df['input_ids'] = it_ai_human_df['input_ids'].apply(ast.literal_eval)
it_ai_human_df['attention_mask'] = it_ai_human_df['attention_mask'].apply(ast.literal_eval)

In [16]:
# Split dataframe
train_df = it_ai_human_df[it_ai_human_df['split'] == 'train']
val_df = it_ai_human_df[it_ai_human_df['split'] == 'val']
test_df = it_ai_human_df[it_ai_human_df['split'] == 'test']

In [17]:
# Initialize dataset
train_dataset = dataset.AIDetectionDataset(train_df)
val_dataset = dataset.AIDetectionDataset(val_df)
test_dataset = dataset.AIDetectionDataset(test_df)

In [18]:
# Check length of each dataset
print('Length of train dataset:     ', len(train_dataset))
print('Length of validation dataset: ', len(val_dataset))
print('Length of test dataset:       ', len(test_dataset))

Length of train dataset:      3502
Length of validation dataset:  732
Length of test dataset:        766


In [19]:
# Test returned types
train_dataset[0]

{'input_ids': tensor([   0, 1222, 7572,  ...,    1,    1,    1]),
 'attention_mask': tensor([1, 1, 1,  ..., 0, 0, 0]),
 'labels': tensor(1)}

## **6. Create Data Loader**

In [20]:
train_loader = data_loader.create_dataloader(train_dataset, batch_size=4)
val_loader = data_loader.create_dataloader(val_dataset, batch_size=4, shuffle=False)
test_loader = data_loader.create_dataloader(test_dataset, batch_size=4, shuffle=False)

In [21]:
# Test case
batch = next(iter(train_loader))
# Check shape
print("Input ID shape:          ", batch['input_ids'].shape)
print("Attention masked's shape:", batch['attention_mask'].shape)
print("Label's shape:           ", batch['labels'].shape)

Input ID shape:           torch.Size([4, 1024])
Attention masked's shape: torch.Size([4, 1024])
Label's shape:            torch.Size([4])


In [22]:
# Check keys
print("Batch's keys:", batch.keys())

Batch's keys: dict_keys(['input_ids', 'attention_mask', 'labels'])


## **7. Define Model**

In [23]:
# Choose device to run model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.AIContentModel(model_name='Qualcomm-AI-Research/BamiBERT', num_classes=2)
model.to(device)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: Qualcomm-AI-Research/BamiBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


AIContentModel(
  (bert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(20481, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(2050, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerN

## **8. Training Setup**

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3 

## **9. Create Training & Validation Functions**

In [25]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc='Training', leave=False, position=1)

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Accuracy': f'{correct/total:.4f}'
        })

    return total_loss / len(loader), correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        progress_bar = tqdm(loader, desc='Validation', leave=False, position=1)

        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Accuracy': f'{correct/total:.4f}'
            })
            
    return total_loss / len(loader), correct / total

## **10. Run Training**

In [ ]:
torch.cuda.empty_cache()
best_val_acc = 0.0
checkpoint_path = '../models/best_model.pt'
start_epoch = 0

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    print(f"Resume from epoch {checkpoint['epoch']}, best_val_acc = {best_val_acc:.4f}")
else:
    print('No checkpoint found, train from scratch')

for epoch in tqdm(range(start_epoch, num_epochs), desc='Epochs'):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f'Epoch {epoch + 1} / {num_epochs}')
    print(f'Train | Loss: {train_loss:.4f} - Accuracy: {train_acc:.4f}')
    print(f'Valid | Loss: {val_loss:.4f} - Accuracy: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            }, checkpoint_path)
        print(f'Save new best model (val_acc = {val_acc:.4f})')